#Carbon Bombs, Analyzing the Potential Exposure to Climatic Extremes Induced by the Largest Fossil Fuel
Projects

Environmental Programming 2026-2027

Group: Ping Pong - Assignment 2

##Setting up environment and importing packages and libraries

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import pickle
import matplotlib as mplt
import os
import cartopy as ctpy
import geopandas as gp
import sys
import scipy
import regionmask as rm
import textwrap

##Data Preprocessing
(tasks 4-5)

In [2]:
#working with laridon excel
dfcoal = pd.read_excel("../data/fossil_fuel_megaprojects_co2_dataset_laridon_2025.xlsx", sheet_name="coal_combustion", header=1)
dfoilgas = pd.read_excel("../data/fossil_fuel_megaprojects_co2_dataset_laridon_2025.xlsx", sheet_name="oil&gas_combustion")

dfcoalmain = dfcoal[["Country", "Project Name", "Best estimate", "P2.5", "P97.5"]] #selects the needed colmuns below set header row
dfcoalmain = dfcoalmain.rename(columns={
    "Country": "country",
    "Project Name": "project",
    "Best estimate": "potential_emissions_best",
    "P2.5": "potential_emissions_p25",
    "P97.5": "potential_emissions_p975",
})
dfcoalmain["fuel"] = "coal"

dfoilgasmain = dfoilgas.iloc[2:, [0,1,3,4,5]].copy() #selects and copies needed columns, ignoring the 2 rows with mixed header information
dfoilgasmain.columns = ["project", "country", "potential_emissions_best", "potential_emissions_p25", "potential_emissions_p975"] #adding new column names
dfoilgasmain = dfoilgasmain.reset_index(drop=True)
dfoilgasmain["fuel"] = "oil-gas"

#combining data in one df, checking number of rows
df = pd.concat([dfcoalmain, dfoilgasmain], ignore_index=True)
print(len(dfcoal))
print(len(dfcoalmain))
print(len(dfoilgas))
print(len(dfoilgasmain))
print(len(dfcoalmain) + len(dfoilgasmain) == len(df))

#assigning correct data types and checking
df["potential_emissions_best"] = df["potential_emissions_best"].astype(float)
df["potential_emissions_p25"] = df["potential_emissions_p25"].astype(float)
df["potential_emissions_p975"] = df["potential_emissions_p975"].astype(float)
print(df.dtypes)

#working with Kuhne excel
dfkuhne = pd.read_excel("../data/carbon_bombs_co2_dataset_kuhne_2022.xlsx", sheet_name="Full Carbon Bombs List")
dfkuhne["new"] = np.where(dfkuhne["New"] == "*", 'yes', 'no')
dfkuhne["id"] = range(1, len(dfkuhne) + 1)
dfkuhne = dfkuhne.rename(columns={"Name": "project"})
dfkuhne = dfkuhne[["id", "new", "project"]]
print(len(dfkuhne))

#setting up final combined data with new index
df_merged = pd.merge(df, dfkuhne, on="project", how="outer")
df_merged = df_merged.set_index("project")

#creating dataset
ds_carbon_bombs = xr.Dataset.from_dataframe(df_merged[["country", "fuel", "new"]])
ds_carbon_bombs = ds_carbon_bombs.set_coords(["country", "fuel"])

#adding variables
ds_carbon_bombs["potential_emissions"] = xr.DataArray(
    df_merged[["potential_emissions_p25", "potential_emissions_p975", "potential_emissions_best"]].values,
    dims=["project", "estimate"],
    coords={"project": df_merged.index, "estimate": ["low", "high", "best"]}
)

ds_carbon_bombs["total"] = len(df_merged)
ds_carbon_bombs["total_oil_and_gas"] = (df_merged["fuel"] == "oil-gas").sum()
ds_carbon_bombs["total_coal"] = (df_merged["fuel"] == "coal").sum()
ds_carbon_bombs["total_new_carbon_bombs"] = (df_merged["new"] == "yes").sum()

#adding attributes for the main variables
ds_carbon_bombs.attrs["description"] = "Carbon bombs project dataset with potential CO2 emissions per project"
ds_carbon_bombs["new"].attrs["description"] = "Indicates whether project was already in Kuhne database or not"
ds_carbon_bombs["potential_emissions"].attrs["description"] = "Provides different estimates for project CO2 emissions, a low estimate for the P2.5, a high estimate for the P97.5, and a best estimate"
ds_carbon_bombs["potential_emissions"].attrs["units"] = "GtCO2"
ds_carbon_bombs["fuel"].attrs["description"] = "Indicates type of project based on Laridon database"

print(ds_carbon_bombs)

pd.to_pickle(ds_carbon_bombs, "../data/ds_carbon_bombs.pkl")

#to reload or check: ds_carbon_bombs = pd.read_pickle("../data/ds_carbon_bombs.pkl") and then print(ds_carbon_bombs)


231
231
197
195
True
country                         str
project                         str
potential_emissions_best    float64
potential_emissions_p25     float64
potential_emissions_p975    float64
fuel                            str
dtype: object
429
<xarray.Dataset> Size: 26kB
Dimensions:                 (project: 455, estimate: 3)
Coordinates:
  * project                 (project) object 4kB 'ACG (Azeri-Chirag-Guneshli ...
    country                 (project) object 4kB 'Azerbaijan' ... nan
    fuel                    (project) object 4kB 'oil-gas' ... 'oil-gas'
  * estimate                (estimate) <U4 48B 'low' 'high' 'best'
Data variables:
    new                     (project) object 4kB 'no' 'no' 'no' ... 'no' 'no'
    potential_emissions     (project, estimate) float64 11kB 1.481 ... 566.5
    total                   int64 8B 455
    total_oil_and_gas       int64 8B 220
    total_coal              int64 8B 234
    total_new_carbon_bombs  int64 8B 176
Attributes:
    descri

Question 6

Question 7